# FanGraphs Batting Leaderboard

This notebook fetches a FanGraphs batting leaderboard and demonstrates filtering,
mutating, and ranking with native Polars.

> **Note:** FanGraphs uses Cloudflare anti-bot protection.
> If you hit a 403, set the `CF_CLEARANCE` environment variable or
> use `BaseballContext(extra_headers={"Cookie": "..."})`.
> See [docs](https://github.com/nicko4o/polars-baseball) for details.

In [ ]:
from __future__ import annotations

import polars as pl

import polars_baseball as pb

## 1. Fetch the 2026 batting leaderboard

Qualification threshold = 100 plate appearances, top 50 results.

In [ ]:
df = await pb.fangraphs.batting(
    start_season=2026,
    end_season=2026,
    qual=100,
    max_results=50,
)
print(f"Rows: {df.height}, Columns: {len(df.columns)}")
df.head(5)

## 2. Top 10 by wRC+

wRC+ (weighted Runs Created Plus) is park-adjusted and league-adjusted.
100 is league average; higher is better.

In [ ]:
top_wrc = (
    df.filter(pl.col("wRC+").is_not_null())
    .select(["Name", "Team", "G", "PA", "HR", "AVG", "OBP", "SLG", "wRC+"])
    .sort("wRC+", descending=True)
    .head(10)
)
top_wrc

## 3. Power hitters: HR leaders

Filter the top 10 home-run hitters with their slugging percentage.

In [ ]:
hr_leaders = (
    df.filter(pl.col("HR") > 0).select(["Name", "Team", "HR", "SLG", "ISO"]).sort("HR", descending=True).head(10)
)
hr_leaders

## 4. Plate discipline leaders

Best K%/BB% ratios among qualified hitters.

Lower K% and higher BB% indicate better plate discipline.

In [ ]:
discipline = (
    df.filter(pl.col("K%").is_not_null() & pl.col("BB%").is_not_null())
    .with_columns((pl.col("BB%") / pl.col("K%")).round(2).alias("BB/K_ratio"))
    .select(["Name", "Team", "K%", "BB%", "BB/K_ratio", "wRC+"])
    .sort("BB/K_ratio", descending=True)
    .head(10)
)
discipline

## 5. Search for a specific team

Filter by team abbreviation (e.g. `LAD` for Dodgers).

In [ ]:
team = "LAD"
team_df = df.filter(pl.col("Team") == team).select(["Name", "G", "PA", "HR", "AVG", "OBP", "SLG", "wRC+"])
print(f"{team} qualified hitters: {team_df.height}")
team_df

## Summary

- FanGraphs leaderboards are fetched as `polars.DataFrame` — ready for analysis.
- Columns like `wRC+`, `K%`, `BB%`, `ISO` are sabermetric staples.
- The same pattern works for `pb.fangraphs.pitching()` and `pb.fangraphs.fielding()`.

Next steps:
- Try `pb.fangraphs.team_batting(start_season=2026, end_season=2026)`
- See [notebooks/mlb_schedule_demo.ipynb](mlb_schedule_demo.ipynb)